# 🔍 FakeScope — 4차 시도

### 변경사항
- Fake에 **Midjourney 실사 AI 이미지** 추가
- Fake에 **AI vs Human (CSV)** 추가
- 균형 맞출 때 얼굴 데이터에서만 자르도록 개선

> **런타임 → GPU 선택**

In [ ]:
# 0. GPU 확인
import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else '없음 ⚠️')

In [ ]:
# 1. Drive 마운트
from google.colab import drive
drive.mount('/content/drive')
import os
os.makedirs('/content/drive/MyDrive/fakescope_checkpoints', exist_ok=True)
!ls /content/drive/MyDrive/fakescope_checkpoints/
print('Drive 마운트 완료!')

In [ ]:
# 2. 패키지 설치
!pip install -q torch torchvision flask numpy opencv-python pillow scikit-learn matplotlib kaggle

In [ ]:
# 3. Kaggle 설정
from google.colab import files
import os, shutil
print('kaggle.json 업로드하세요:')
uploaded = files.upload()
os.makedirs('/root/.kaggle', exist_ok=True)
shutil.copy('kaggle.json', '/root/.kaggle/kaggle.json')
os.chmod('/root/.kaggle/kaggle.json', 0o600)
print('완료!')

In [ ]:
# 4-1. 기존 데이터셋 다운로드
import os
for d in ['/content/data/faces', '/content/data/pixiv',
          '/content/data/intel', '/content/data/ai_generated',
          '/content/data/coco', '/content/data/midjourney',
          '/content/data/aivshuman']:
    os.makedirs(d, exist_ok=True)

print('① 140k 얼굴...')
!kaggle datasets download -d xhlulu/140k-real-and-fake-faces -p /content/data/faces/
!unzip -q /content/data/faces/140k-real-and-fake-faces.zip -d /content/data/faces/

print('② Pixiv...')
!kaggle datasets download -d tagamit/pixiv-aireal-art -p /content/data/pixiv/
!unzip -q /content/data/pixiv/pixiv-aireal-art.zip -d /content/data/pixiv/

print('③ Intel...')
!kaggle datasets download -d puneet6060/intel-image-classification -p /content/data/intel/
!unzip -q /content/data/intel/intel-image-classification.zip -d /content/data/intel/

print('④ AI Generated Images...')
!kaggle datasets download -d aloktantrik/a-dataset-of-34500-labeled-images -p /content/data/ai_generated/
!unzip -q /content/data/ai_generated/a-dataset-of-34500-labeled-images.zip -d /content/data/ai_generated/

print('모두 완료!')

In [ ]:
# 4-2. COCO 다운로드
print('⑤ COCO train2017 (약 18GB)...')
os.makedirs('/content/data/coco/coco_data/images', exist_ok=True)
!wget -q http://images.cocodataset.org/zips/train2017.zip -O /content/data/coco/train2017.zip
!unzip -q /content/data/coco/train2017.zip -d /content/data/coco/coco_data/images/
print('COCO 완료!')

In [ ]:
# 4-3. Midjourney 다운로드
print('⑥ Midjourney Images...')
!kaggle datasets download -d cyanex1702/midjourney-imagesprompt -p /content/data/midjourney/
!unzip -q /content/data/midjourney/midjourney-imagesprompt.zip -d /content/data/midjourney/
!find /content/data/midjourney/ -type d | head -10
print('Midjourney 완료!')

In [ ]:
# 4-4. AI vs Human (CSV) 다운로드
print('⑦ AI vs Human Generated Dataset...')
!kaggle datasets download -d alessandrasala79/ai-vs-human-generated-dataset -p /content/data/aivshuman/
!unzip -q /content/data/aivshuman/ai-vs-human-generated-dataset.zip -d /content/data/aivshuman/
!find /content/data/aivshuman/ -type f -name '*.csv' | head -5
print('AI vs Human 완료!')

In [ ]:
# 5. CSV 구조 확인 (경로 설정 전 필수)
import pandas as pd

# CSV 파일 경로 확인
!find /content/data/aivshuman/ -name '*.csv'

# CSV 내용 확인
df = pd.read_csv('/content/data/aivshuman/train_data/train.csv')  # 실제 경로로 수정
print(df.head())
print('컬럼:', df.columns.tolist())
print('레이블 분포:', df['label'].value_counts())

In [ ]:
# 6. 경로 설정 (위 셀 결과 보고 수정)
FACES_DIR   = '/content/data/faces/real_vs_fake/real-vs-fake'
PIXIV_DIR   = '/content/data/pixiv/aidataset'
INTEL_DIR   = '/content/data/intel'
AI_GEN_DIR  = '/content/data/ai_generated/labeled_images'
COCO_DIR    = '/content/data/coco'
MJ_DIR      = '/content/data/midjourney/images'  # 실제 폴더명으로 수정
CSV_PATH    = '/content/data/aivshuman/train_data/train.csv'  # 실제 경로로 수정
CSV_IMG_DIR = '/content/data/aivshuman/train_data'            # 실제 경로로 수정

for name, path in [('FACES', FACES_DIR), ('PIXIV', PIXIV_DIR),
                   ('INTEL', INTEL_DIR), ('AI_GEN', AI_GEN_DIR),
                   ('COCO', COCO_DIR), ('MJ', MJ_DIR),
                   ('CSV', CSV_PATH)]:
    import os
    exists = '✅' if os.path.exists(path) else '❌ 없음!'
    print(f'{name:8s}: {exists} {path}')

In [ ]:
# 7. 프로젝트 코드 업로드
from google.colab import files
import os, shutil, sys

os.makedirs('/content/fakescope/src', exist_ok=True)
os.chdir('/content/fakescope')
sys.path.insert(0, '/content/fakescope/src')

print('dataset.py, model.py, train.py, gradcam.py 업로드하세요:')
uploaded = files.upload()
for fname in uploaded:
    shutil.move(fname, f'src/{fname}')
    print(f'  이동: src/{fname}')

In [ ]:
# 8. 데이터 로딩 테스트
from dataset import get_dataloaders

loaders = get_dataloaders(
    faces_dir   = FACES_DIR,
    pixiv_dir   = PIXIV_DIR,
    intel_dir   = INTEL_DIR,
    ai_gen_dir  = AI_GEN_DIR,
    coco_dir    = COCO_DIR,
    mj_dir      = MJ_DIR,
    csv_path    = CSV_PATH,
    csv_img_dir = CSV_IMG_DIR,
    batch_size  = 32,
    coco_sample = 10000,
)

imgs, labels = next(iter(loaders['train']))
print(f'배치 shape: {imgs.shape}')
print(f'Real: {(labels==0).sum().item()}장, Fake: {(labels==1).sum().item()}장')

In [ ]:
# 9. 학습
import argparse
from train import train

args = argparse.Namespace(
    faces_dir   = FACES_DIR,
    pixiv_dir   = PIXIV_DIR,
    intel_dir   = INTEL_DIR,
    ai_gen_dir  = AI_GEN_DIR,
    coco_dir    = COCO_DIR,
    mj_dir      = MJ_DIR,
    csv_path    = CSV_PATH,
    csv_img_dir = CSV_IMG_DIR,
    weights_dir = '/content/fakescope/weights',
    use_fft     = True,
    epochs      = 30,
    batch_size  = 32,
    lr          = 1e-4,
    patience    = 5,
    coco_sample = 10000,
)

history = train(args)

In [ ]:
# 10. 학습 곡선
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
epochs = range(1, len(history['train_loss']) + 1)

ax1.plot(epochs, history['train_loss'], label='Train Loss')
ax1.plot(epochs, history['val_loss'],   label='Val Loss')
ax1.set_title('Loss'); ax1.set_xlabel('Epoch')
ax1.legend(); ax1.grid(alpha=.3)

ax2.plot(epochs, history['train_acc'], label='Train Acc')
ax2.plot(epochs, history['val_acc'],   label='Val Acc')
ax2.set_title('Accuracy'); ax2.set_xlabel('Epoch')
ax2.legend(); ax2.grid(alpha=.3)

plt.tight_layout()
plt.savefig('training_curve.png', dpi=150)
plt.show()

In [ ]:
# 11. 다운로드
from google.colab import files
import shutil

shutil.copy(
    '/content/drive/MyDrive/fakescope_checkpoints/best_model.pth',
    '/content/best_model.pth'
)
files.download('/content/best_model.pth')
files.download('/content/fakescope/training_curve.png')
print('다운로드 완료!')